# Tests: `fasterai.misc.bn_folding` (source `nbs/misc/bn_folding.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fasterai.misc.bn_folding import *

In [ ]:
from fastcore.test import *

# Fold preserves forward pass
model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32)
).eval()
x = torch.randn(2, 3, 8, 8)

with torch.no_grad():
    out_orig = model(x)
    folder = BN_Folder()
    model_f = folder.fold(model)
    out_fold = model_f(x)

test_close(out_orig, out_fold, eps=1e-5)

# BN layers replaced with Identity after folding
bn_count = sum(1 for m in model_f.modules() if isinstance(m, nn.BatchNorm2d))
test_eq(bn_count, 0)

# Folded conv has bias (even if original didn't)
assert model_f[0].bias is not None
assert model_f[3].bias is not None

# Identity modules present where BN was
id_count = sum(1 for m in model_f.modules() if isinstance(m, nn.Identity))
test_eq(id_count, 2)

In [ ]:
def _n_subnormal(t):
    tiny = torch.finfo(t.dtype).smallest_normal
    return int(((t.abs() > 0) & (t.abs() < tiny)).sum())

def _collapse_bn(bn, dead):
    "Force the statistics and the beta of the dead channels into the subnormal range"
    with torch.no_grad():
        bn.running_mean[dead], bn.running_var[dead], bn.bias[dead] = 5.6e-45, 5.6e-45, 1e-40

TINY32 = torch.finfo(torch.float32).smallest_normal

# Channels emptied by sparsification: a subnormal beta must not reach the folded bias
torch.manual_seed(0)
conv, bn = nn.Conv2d(4, 4, 3), nn.BatchNorm2d(4)
dead = torch.tensor([True, True, True, False])
with torch.no_grad():
    conv.weight[dead] = 0.; conv.bias[dead] = 0.
    _collapse_bn(bn, dead)
    bn.bias[1] = -1e-40                                                # negative subnormals go too
    bn.running_mean[2], bn.running_var[2], bn.bias[2] = 0., 1., TINY32 # the smallest normal stays
model = nn.Sequential(conv, bn).eval()
folded = BN_Folder().fold(model)

for p in folded.parameters(): test_eq(_n_subnormal(p.data), 0)
test_eq(folded[0].bias.data[0].item(), 0.)
test_eq(folded[0].bias.data[1].item(), 0.)
test_eq(folded[0].bias.data[2].item(), TINY32)

x = torch.randn(2, 4, 8, 8)
with torch.no_grad(): test_close(model(x), folded(x), eps=1e-6)

# The bias-less conv takes the `conv_b is None` path, which writes a folded bias too
conv, bn = nn.Conv2d(4, 4, 3, bias=False), nn.BatchNorm2d(4)
with torch.no_grad():
    conv.weight[0] = 0.
    _collapse_bn(bn, torch.tensor([True, False, False, False]))
folded = BN_Folder().fold(nn.Sequential(conv, bn).eval())

for p in folded.parameters(): test_eq(_n_subnormal(p.data), 0)
test_eq(folded[0].bias.data[0].item(), 0.)

In [ ]:
# Healthy channels: the fold stays bit-identical to the plain formula
torch.manual_seed(1)
conv, bn = nn.Conv2d(8, 8, 3), nn.BatchNorm2d(8)
with torch.no_grad():
    bn.running_mean.uniform_(-1, 1); bn.running_var.uniform_(0.5, 1.5)
    bn.weight.uniform_(0.5, 1.5); bn.bias.uniform_(-1, 1)
folded = BN_Folder().fold(nn.Sequential(conv, bn).eval())

rsqrt = torch.rsqrt(bn.running_var + bn.eps)
w_ref = conv.weight.data * (bn.weight.data * rsqrt).view(-1, 1, 1, 1)
b_ref = (conv.bias.data - bn.running_mean) * rsqrt * bn.weight.data + bn.bias.data

test_eq(_n_subnormal(w_ref) + _n_subnormal(b_ref), 0)
assert torch.equal(folded[0].weight.data, w_ref)
assert torch.equal(folded[0].bias.data, b_ref)

In [ ]:
# Half precision is left alone: its smallest normal is 6.1e-05, a value a fold cannot neglect
torch.manual_seed(2)
conv, bn = nn.Conv2d(4, 4, 3).half(), nn.BatchNorm2d(4).half()
with torch.no_grad():
    conv.weight[0] = 0.; conv.bias[0] = 0.
    bn.running_mean[0], bn.bias[0] = 0., 1e-5      # below float16's smallest normal
folded = BN_Folder().fold(nn.Sequential(conv, bn).eval())

rsqrt = torch.rsqrt(bn.running_var + bn.eps)
w_ref = conv.weight.data * (bn.weight.data * rsqrt).view(-1, 1, 1, 1)
b_ref = (conv.bias.data - bn.running_mean) * rsqrt * bn.weight.data + bn.bias.data

test_eq(_n_subnormal(folded[0].bias.data), 1)
assert torch.equal(folded[0].weight.data, w_ref)
assert torch.equal(folded[0].bias.data, b_ref)

In [ ]:
#| slow
from torchvision.models import resnet18
from fasterai.core.criteria import large_final
from fasterai.sparse.sparsifier import Sparsifier

def _model_subnormals(m):
    return sum(_n_subnormal(t) for t in list(m.parameters()) + [b for b in m.buffers() if b.is_floating_point()])

# A sparsified ResNet-18: dead filters, with BN statistics forced into the subnormal range
torch.manual_seed(0)
model = resnet18(weights=None).eval()
Sparsifier(model, 'filter', 'local', large_final).sparsify_model(0.5)
for bn in model.modules():
    if isinstance(bn, nn.BatchNorm2d): _collapse_bn(bn, bn.weight.data == 0)

x = torch.randn(1, 3, 224, 224)
folded = BN_Folder().fold(model.eval())
test_eq(_model_subnormals(folded), 0)
with torch.no_grad(): test_eq(model(x).argmax(1), folded(x).argmax(1))